# 01 - 数据准备和特征工程

在本笔记本中，我们将：
1.从HuggingFace加载预处理后的数据集
2.应用增强特征工程
3. 生成文本嵌入
4.保存增强数据集用于模型训练

In [ ]:
# 导入依赖
# Imports
import sys
sys.path.append('../')

import os
from dotenv import load_dotenv
from huggingface_hub import login
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

from src.items import Item
from src.features import FeatureEngineer
from src.embeddings import EmbeddingGenerator, batch_embed_items

load_dotenv(override=True)

# 登录 HuggingFace
# Login to HuggingFace
hf_token = os.environ.get('HF_TOKEN')
if hf_token:
    login(hf_token, add_to_git_credential=True)

## 步骤 1：加载数据集

我们将使用 Ed 的精简数据集来加快实验速度

In [ ]:
# 加载预处理的数据集
# Load the preprocessed dataset
username = "ed-donner"
dataset_name = f"{username}/items_lite"

print(f"Loading dataset: {dataset_name}")
train, val, test = Item.from_hub(dataset_name)

print(f"\nDataset loaded successfully!")
print(f"Training items: {len(train):,}")
print(f"Validation items: {len(val):,}")
print(f"Test items: {len(test):,}")
print(f"Total items: {len(train) + len(val) + len(test):,}")

In [ ]:
# 检查样品项目
# Inspect a sample item
print("Sample item:")
print(train[0])
print(f"\nSummary: {train[0].summary}")
print(f"Price: ${train[0].price}")

## 步骤 2：增强特征工程

从产品数据中提取附加特征

In [ ]:
# 将增强功能应用于所有项目
# Apply enhancements to all items
print("Enhancing training items...")
for item in tqdm(train):
    item.enhance()

print("\nEnhancing validation items...")
for item in tqdm(val):
    item.enhance()

print("\nEnhancing test items...")
for item in tqdm(test):
    item.enhance()

print("\nFeature engineering complete!")

In [ ]:
# 检查增强功能
# Inspect enhanced features
sample = train[0]
print("Enhanced features:")
print(f"Brand: {sample.brand}")
print(f"Word count: {sample.word_count}")
print(f"Character count: {sample.char_count}")
print(f"Has dimensions: {sample.has_dimensions}")
print(f"Is premium: {sample.is_premium}")
print(f"Text quality score: {sample.text_quality_score:.3f}")
print(f"Price bucket: {sample.price_bucket}")
print(f"Price per weight: {sample.price_per_weight}")

## 步骤 3：可视化增强功能

In [ ]:
# 按桶划分的价格分布
# Price distribution by bucket
from collections import Counter

bucket_counts = Counter([item.price_bucket for item in train])

plt.figure(figsize=(12, 6))
plt.bar(bucket_counts.keys(), bucket_counts.values(), color='skyblue')
plt.title('Price Distribution by Bucket')
plt.xlabel('Price Bucket')
plt.ylabel('Count')
plt.xticks(rotation=45)
for i, (k, v) in enumerate(bucket_counts.items()):
    plt.text(i, v, f"{v:,}", ha='center', va='bottom')
plt.tight_layout()
plt.show()

In [ ]:
# 溢价与非溢价发行
# Premium vs non-premium distribution
premium_count = sum(1 for item in train if item.is_premium)
non_premium_count = len(train) - premium_count

plt.figure(figsize=(8, 8))
plt.pie([premium_count, non_premium_count], 
        labels=['Premium', 'Non-Premium'],
        autopct='%1.1f%%',
        colors=['gold', 'lightblue'])
plt.title('Premium vs Non-Premium Products')
plt.show()

## 步骤 4：生成文本嵌入

使用句子转换器创建语义嵌入

In [ ]:
# 为所有数据集生成嵌入
# Generate embeddings for all datasets
print("Generating embeddings...\n")

# 如果数据目录不存在则创建
# Create data directory if it doesn't exist
os.makedirs('../data/embeddings', exist_ok=True)

# 生成并缓存嵌入
# Generate and cache embeddings
train_embeddings = batch_embed_items(
    train, 
    model_name='all-MiniLM-L6-v2',
    cache_path='../data/embeddings/train_embeddings.npy'
)

val_embeddings = batch_embed_items(
    val,
    model_name='all-MiniLM-L6-v2', 
    cache_path='../data/embeddings/val_embeddings.npy'
)

test_embeddings = batch_embed_items(
    test,
    model_name='all-MiniLM-L6-v2',
    cache_path='../data/embeddings/test_embeddings.npy'
)

print(f"\nEmbeddings generated!")
print(f"Train embeddings shape: {train_embeddings.shape}")
print(f"Val embeddings shape: {val_embeddings.shape}")
print(f"Test embeddings shape: {test_embeddings.shape}")

In [ ]:
# 将嵌入存储在项目中以供以后使用
# Store embeddings in items for later use
for item, emb in zip(train, train_embeddings):
    item.embedding = emb.tolist()

for item, emb in zip(val, val_embeddings):
    item.embedding = emb.tolist()

for item, emb in zip(test, test_embeddings):
    item.embedding = emb.tolist()

print("Embeddings stored in items!")

## 步骤 5：提取 ML 模型的特征

In [ ]:
# 初始化特征工程师
# Initialize feature engineer
feature_engineer = FeatureEngineer()

# 拟合和转换训练数据
# Fit and transform training data
train_features = feature_engineer.fit_transform(train)
val_features = feature_engineer.transform(val)
test_features = feature_engineer.transform(test)

print("Feature extraction complete!")
print(f"\nTraining features shape: {train_features.shape}")
print(f"Validation features shape: {val_features.shape}")
print(f"Test features shape: {test_features.shape}")
print(f"\nFeature columns: {list(train_features.columns)}")

In [ ]:
# 显示特征统计数据
# Display feature statistics
train_features.describe()

## 步骤 6：保存增强数据

In [ ]:
# 保存功能
# Save features
train_features.to_csv('../data/train_features.csv', index=False)
val_features.to_csv('../data/val_features.csv', index=False)
test_features.to_csv('../data/test_features.csv', index=False)

# 节省价格
# Save prices
train_prices = np.array([item.price for item in train])
val_prices = np.array([item.price for item in val])
test_prices = np.array([item.price for item in test])

np.save('../data/train_prices.npy', train_prices)
np.save('../data/val_prices.npy', val_prices)
np.save('../data/test_prices.npy', test_prices)

print("Data saved successfully!")
print("\nFiles created:")
print("- data/train_features.csv")
print("- data/val_features.csv")
print("- data/test_features.csv")
print("- data/train_prices.npy")
print("- data/val_prices.npy")
print("- data/test_prices.npy")
print("- data/embeddings/train_embeddings.npy")
print("- data/embeddings/val_embeddings.npy")
print("- data/embeddings/test_embeddings.npy")

## 概括

✅ 从 HuggingFace 加载数据集  
✅ 应用增强特征工程  
✅ 生成的文本嵌入  
✅ 为 ML 模型提取特征  
✅ 保存后续步骤的所有数据

**下一篇：** 笔记本02 - 构建RAG系统